# Kong_HW3



In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score
)


In [2]:

feature_names = [
    "word_freq_make","word_freq_address","word_freq_all","word_freq_3d","word_freq_our","word_freq_over","word_freq_remove",
    "word_freq_internet","word_freq_order","word_freq_mail","word_freq_receive","word_freq_will","word_freq_people","word_freq_report",
    "word_freq_addresses","word_freq_free","word_freq_business","word_freq_email","word_freq_you","word_freq_credit","word_freq_your",
    "word_freq_font","word_freq_000","word_freq_money","word_freq_hp","word_freq_hpl","word_freq_george","word_freq_650","word_freq_lab",
    "word_freq_labs","word_freq_telnet","word_freq_857","word_freq_data","word_freq_415","word_freq_85","word_freq_technology","word_freq_1999",
    "word_freq_parts","word_freq_pm","word_freq_direct","word_freq_cs","word_freq_meeting","word_freq_original","word_freq_project","word_freq_re",
    "word_freq_edu","word_freq_table","word_freq_conference","char_freq_;","char_freq_(","char_freq_[","char_freq_!","char_freq_$","char_freq_#",
    "capital_run_length_average","capital_run_length_longest","capital_run_length_total","spam"
]

df = pd.read_csv("spambase.data", header=None, names=feature_names)
X = df.drop(columns=["spam"]).values
y = df["spam"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

def metric_dict(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    return {
        "accuracy": acc,
        "error": 1 - acc,
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }

print(df.shape)


FileNotFoundError: [Errno 2] No such file or directory: 'spambase.data'

In [ ]:

# Problem 1: package logistic regression
logreg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, solver="liblinear", random_state=42))
])
logreg_pipe.fit(X_train, y_train)

y_pred_test = logreg_pipe.predict(X_test)
y_prob_test = logreg_pipe.predict_proba(X_test)[:, 1]

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_test))
print("\nTest metrics:")
print(metric_dict(y_test, y_pred_test))

coef_series = pd.Series(logreg_pipe.named_steps["clf"].coef_[0], index=feature_names[:-1])
print("\nTop positive coefficients:")
print(coef_series.sort_values(ascending=False).head(10))
print("\nTop negative coefficients:")
print(coef_series.sort_values().head(10))

for t in [0.25, 0.5, 0.75, 0.9]:
    pred_t = (y_prob_test >= t).astype(int)
    print(f"\nThreshold = {t}")
    print({
        "accuracy": accuracy_score(y_test, pred_t),
        "precision": precision_score(y_test, pred_t, zero_division=0),
        "recall": recall_score(y_test, pred_t, zero_division=0)
    })


In [ ]:

# Problem 2: manual gradient descent for logistic regression

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def add_intercept(X):
    return np.c_[np.ones((X.shape[0], 1)), X]

scaler_manual = StandardScaler()
X_train_s = scaler_manual.fit_transform(X_train)
X_test_s = scaler_manual.transform(X_test)
Xtr_i = add_intercept(X_train_s)
Xte_i = add_intercept(X_test_s)

def logistic_gd(X, y, lr, n_iter=100):
    n, d = X.shape
    w = np.zeros(d)
    checkpoints = {}
    for it in range(1, n_iter + 1):
        p = sigmoid(X @ w)
        grad = X.T @ (p - y) / n
        w -= lr * grad
        if it in [10, 50, 100]:
            loss = -np.mean(y * np.log(np.clip(p, 1e-12, 1 - 1e-12)) + (1 - y) * np.log(np.clip(1 - p, 1e-12, 1 - 1e-12)))
            checkpoints[it] = loss
    return w, checkpoints

def evaluate_w(X, y, w):
    p = sigmoid(X @ w)
    yp = (p >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(y, yp),
        "precision": precision_score(y, yp),
        "recall": recall_score(y, yp),
        "f1": f1_score(y, yp),
    }

for lr in [0.01, 0.1, 1.0]:
    w, checkpoints = logistic_gd(Xtr_i, y_train, lr=lr, n_iter=100)
    print("\nLearning rate:", lr)
    print("Loss checkpoints:", checkpoints)
    print("Train metrics:", evaluate_w(Xtr_i, y_train, w))
    print("Test metrics:", evaluate_w(Xte_i, y_test, w))


In [ ]:

# Problem 3: compare LR, LDA, and kNN
candidate_k = [1, 3, 5, 7, 9, 11, 15]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_rows = []
for k in candidate_k:
    fold_errors = []
    fold_acc = []
    fold_prec = []
    fold_rec = []
    for tr_idx, val_idx in skf.split(X_train, y_train):
        Xtr, Xval = X_train[tr_idx], X_train[val_idx]
        ytr, yval = y_train[tr_idx], y_train[val_idx]
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("knn", KNeighborsClassifier(n_neighbors=k, algorithm="kd_tree"))
        ])
        pipe.fit(Xtr, ytr)
        yp = pipe.predict(Xval)
        acc = accuracy_score(yval, yp)
        fold_acc.append(acc)
        fold_errors.append(1 - acc)
        fold_prec.append(precision_score(yval, yp))
        fold_rec.append(recall_score(yval, yp))
    cv_rows.append({
        "k": k,
        "accuracy": np.mean(fold_acc),
        "error": np.mean(fold_errors),
        "precision": np.mean(fold_prec),
        "recall": np.mean(fold_rec),
    })

cv_df = pd.DataFrame(cv_rows)
print(cv_df)

best_k = cv_df.sort_values("error").iloc[0]["k"]
print("\nBest k =", int(best_k))

lda_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LinearDiscriminantAnalysis())
])
lda_pipe.fit(X_train, y_train)

knn_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=int(best_k), algorithm="kd_tree"))
])
knn_pipe.fit(X_train, y_train)

models = {
    "Logistic Regression": logreg_pipe,
    "LDA": lda_pipe,
    "kNN": knn_pipe,
}

comparison_rows = []
for name, model in models.items():
    for split_name, Xs, ys in [("train", X_train, y_train), ("test", X_test, y_test)]:
        yp = model.predict(Xs)
        row = metric_dict(ys, yp)
        row["model"] = name
        row["split"] = split_name
        comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
print("\nClassifier comparison:")
print(comparison_df)


In [ ]:

# Problem 3: ROC with and without a package
fpr, tpr, _ = roc_curve(y_test, y_prob_test)
auc = roc_auc_score(y_test, y_prob_test)
print("AUC =", auc)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"Package ROC (AUC = {auc:.4f})")
plt.plot([0, 1], [0, 1], "--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve for Logistic Regression")
plt.legend()
plt.show()

manual_points = []
for t in np.arange(0, 1.01, 0.1):
    yp = (y_prob_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, yp).ravel()
    fpr_t = fp / (fp + tn)
    tpr_t = tp / (tp + fn)
    manual_points.append((t, fpr_t, tpr_t))

manual_df = pd.DataFrame(manual_points, columns=["threshold", "fpr", "tpr"])
print(manual_df)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label="Package ROC")
plt.plot(manual_df["fpr"], manual_df["tpr"], marker="o", label="Manual ROC")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Manual vs Package ROC")
plt.legend()
plt.show()


In [ ]:

# Problem 4: manual k-fold cross-validation for LR and LDA

def manual_kfold_cv(model_name, X, y, k):
    rng = np.random.default_rng(42)
    idx = np.arange(len(y))
    rng.shuffle(idx)
    folds = np.array_split(idx, k)
    errors = []

    for i in range(k):
        val_idx = folds[i]
        tr_idx = np.concatenate([folds[j] for j in range(k) if j != i])

        Xtr, Xval = X[tr_idx], X[val_idx]
        ytr, yval = y[tr_idx], y[val_idx]

        if model_name == "logreg":
            model = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(max_iter=5000, solver="liblinear", random_state=42))
            ])
        else:
            model = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LinearDiscriminantAnalysis())
            ])

        model.fit(Xtr, ytr)
        yp = model.predict(Xval)
        errors.append(1 - accuracy_score(yval, yp))

    return np.mean(errors), errors

for model_name in ["logreg", "lda"]:
    for k in [5, 10]:
        avg_error, fold_errors = manual_kfold_cv(model_name, X, y, k)
        print(model_name, "k =", k, "average validation error =", avg_error)
        print("fold errors =", fold_errors)
